# Getting started with pyhazrd

`pyhazrd` provides a consistent API for computing survival-based discrimination
metrics and visualizing Kaplan-Meier curves stratified by polygenic hazard score
(PHS). The primary user-facing functions are `phs_metrics()` for statistics and
`phs_km_curve()` for Kaplan-Meier plots.

In [ ]:
import pyreadr
import pandas as pd
import matplotlib.pyplot as plt

from pyhazrd import phs_metrics, phs_km_curve

## Data

The package ships with a simulated survival dataset, `test_data`. The three
required columns are `phs` (continuous score), `age` (time to event or
censoring), and `status` (event indicator: 1 = event, 0 = censored).

In [ ]:
result = pyreadr.read_r("../data/test_data.rda")
test_data = result["test_data"]
test_data.head()

## Computing metrics with `phs_metrics()`

`phs_metrics()` is the single entry point for all discrimination statistics. It
fits a Cox proportional hazards model and derives metrics from the fit. The
`metrics` argument controls which statistics are computed; column names are
passed as strings and default to `"phs"`, `"age"`, and `"status"`.

### All core metrics in one call

In [ ]:
metrics = phs_metrics(
    test_data,
    metrics=["HR", "C_index", "OR", "HR_SD"],
    or_age=70,
)
metrics

The returned DataFrame has one row per metric. `conf_low`, `conf_high`, and `se`
are `NaN` until bootstrapping is enabled.

### Hazard ratio

By default `phs_metrics()` computes `HR[80-100]_[0-20]` — the ratio of the top
20% to the bottom 20% of the PHS distribution. Use `hr_numerator` /
`hr_denominator` to change the bands:

In [ ]:
phs_metrics(test_data, metrics=["HR"], hr_numerator=0.90, hr_denominator=0.10)

For multiple HRs in one call, supply `hr_pairs`:

In [ ]:
phs_metrics(
    test_data,
    metrics=["HR"],
    hr_pairs=[
        {"numerator": (0.80, 1.00), "denominator": (0.00, 0.20)},
        {"numerator": (0.80, 1.00), "denominator": (0.40, 0.60)},
    ],
)

### Odds ratio

The OR is computed from Kaplan-Meier survival estimates at a specific age.
`or_age` accepts a list; one row is returned per age:

In [ ]:
phs_metrics(test_data, metrics=["OR"], or_age=[65, 70, 75])

### Bootstrapped confidence intervals

Set `bootstrap=True` to populate `conf_low`, `conf_high`, and `se`. Use a
small `n_boot` for speed during development:

In [ ]:
phs_metrics(
    test_data,
    metrics=["HR", "C_index", "HR_SD"],
    bootstrap=True,
    n_boot=300,
    seed=42,
)

### Non-default column names

Column names are passed as strings, making `phs_metrics()` straightforward to
use with non-standard DataFrames:

In [ ]:
test_data2 = test_data.rename(columns={
    "phs": "score",
    "age": "diagnosis_age",
    "status": "case",
})

phs_metrics(
    test_data2,
    phs="score",
    time="diagnosis_age",
    event="case",
    metrics=["HR", "C_index"],
)

## Kaplan-Meier curves with `phs_km_curve()`

`phs_km_curve()` stratifies the cohort into percentile bands and plots
empirical Kaplan-Meier survival curves for each band. It returns either a
`matplotlib.figure.Figure` (`output="plot"`, the default) or a tidy DataFrame
(`output="data"`).

### Default plot

The default `breaks=[0.20, 0.80]` splits the cohort into bottom 20%,
middle 60%, and top 20%:

In [ ]:
fig = phs_km_curve(test_data)
plt.show()

### Custom cutpoints

Pass any list of percentile cutpoints (strictly in (0, 1)) to `breaks`:

In [ ]:
# Quintiles
fig = phs_km_curve(test_data, breaks=[0.20, 0.40, 0.60, 0.80])
plt.show()

### Modifying the matplotlib output

`phs_km_curve()` returns a standard `matplotlib.figure.Figure` that can be
customised further:

In [ ]:
fig = phs_km_curve(test_data)
ax = fig.axes[0]
ax.set_title("Disease-free Survival by PHS Percentile")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Disease-free Probability")
ax.legend(title="PHS Group")
ax.set_xlim(40, 100)
plt.show()

### Getting the underlying data

Use `output="data"` when you want to build a fully custom plot or pass the
survival estimates on to further analysis:

In [ ]:
km_data = phs_km_curve(test_data, output="data")
km_data.head()

The returned DataFrame has columns `time`, `estimate`, `conf.low`, `conf.high`,
and `stratum`, which can be used directly with matplotlib:

In [ ]:
fig, ax = plt.subplots()

colors = ["#E41A1C", "#377EB8", "#4DAF4A"]
for i, (stratum, grp) in enumerate(km_data.groupby("stratum", sort=False)):
    color = colors[i % len(colors)]
    ax.step(grp["time"], grp["estimate"], where="post", color=color,
            label=stratum, linewidth=0.8)
    ax.fill_between(grp["time"], grp["conf.low"], grp["conf.high"],
                    step="post", alpha=0.15, color=color)

ax.set_xlim(40, 100)
ax.set_ylim(0, 1)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Disease-free Probability")
ax.legend(title="PHS Group")
plt.show()